In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_path = "/home/thanhdo/hub/deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
device = "cuda:1"
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map={"": device},
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [8]:
messages = [
    {"role": "user", "content": "Hello bro"},
    {"role": "assistant", "content": "Wassup bro"},
    {"role": "user", "content": "What can I do for you bro?"},
    {"role": "assistant", "content": "Anything bro"},
]
bro = tokenizer.apply_chat_template(
    messages,
    tokenize              = False,
    add_generation_prompt = False,
    # return_tensors        = "pt",
)
print(bro)

<｜begin▁of▁sentence｜><｜User｜>Hello bro<｜Assistant｜>Wassup bro<｜end▁of▁sentence｜><｜User｜>What can I do for you bro?<｜Assistant｜>Anything bro<｜end▁of▁sentence｜>


In [9]:
tokenizer.chat_template

"{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% set ns = namespace(is_first=false, is_tool=false, is_output_first=true, system_prompt='') %}{%- for message in messages %}{%- if message['role'] == 'system' %}{% set ns.system_prompt = message['content'] %}{%- endif %}{%- endfor %}{{bos_token}}{{ns.system_prompt}}{%- for message in messages %}{%- if message['role'] == 'user' %}{%- set ns.is_tool = false -%}{{'<｜User｜>' + message['content']}}{%- endif %}{%- if message['role'] == 'assistant' and message['content'] is none %}{%- set ns.is_tool = false -%}{%- for tool in message['tool_calls']%}{%- if not ns.is_first %}{{'<｜Assistant｜><｜tool▁calls▁begin｜><｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['function']['name'] + '\\n' + '```json' + '\\n' + tool['function']['arguments'] + '\\n' + '```' + '<｜tool▁call▁end｜>'}}{%- set ns.is_first = true -%}{%- else %}{{'\\n' + '<｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['fu